In [8]:
import os
import random
import json
import shutil
from tqdm import tqdm

# Paths
SOURCE_DIR = "../../../datasets/chosen_images"
OUTPUT_ROOT = "../../../datasets/CRADataset"
INPUT_JSON = "result_coco.json"

In [9]:
# Settings
SPLIT_RATIOS = {"train": 0.7, "val": 0.2, "test": 0.1}
SEED = 42

# Load input COCO-style annotations
with open(INPUT_JSON, "r") as f:
    data = json.load(f)

# Fix image paths: extract only filename from full path
for img in data["images"]:
    full_path = img["file_name"]
    filename = os.path.basename(full_path)
    img["file_name"] = filename

# Get full list of filenames from COCO JSON that exist in SOURCE_DIR
available_filenames = set(os.listdir(SOURCE_DIR))
coco_images = [img for img in data["images"] if img["file_name"] in available_filenames]

# Reindex image IDs and build mapping
image_id_map = {}
for new_id, img in enumerate(coco_images, 1):
    old_id = img["id"]
    img["id"] = new_id
    image_id_map[old_id] = new_id

# Filter annotations to match the selected images
image_ids_selected = set(image_id_map.keys())
filtered_anns = [
    {**ann, "image_id": image_id_map[ann["image_id"]], "id": i + 1}
    for i, ann in enumerate(data["annotations"])
    if ann["image_id"] in image_ids_selected
]

# Shuffle and split
random.seed(SEED)
random.shuffle(coco_images)
n_total = len(coco_images)
n_train = int(SPLIT_RATIOS["train"] * n_total)
n_val = int(SPLIT_RATIOS["val"] * n_total)

splits = {
    "train": coco_images[:n_train],
    "val": coco_images[n_train:n_train + n_val],
    "test": coco_images[n_train + n_val:]
}

# Create output directories and split-specific annotation files
for split_name, split_images in splits.items():
    print(f"\nProcessing split: {split_name} ({len(split_images)} images)")
    split_dir = os.path.join(OUTPUT_ROOT, split_name)
    os.makedirs(split_dir, exist_ok=True)

    split_image_ids = {img["id"] for img in split_images}
    split_anns = [ann for ann in filtered_anns if ann["image_id"] in split_image_ids]

    # Copy images
    for img in tqdm(split_images):
        src_path = os.path.join(SOURCE_DIR, img["file_name"])
        dst_path = os.path.join(split_dir, img["file_name"])
        shutil.copy2(src_path, dst_path)

    # Clean up: remove 'path' key if present
    for img in split_images:
        img.pop("path", None)  # safely remove if it exists

    # Write new COCO JSON
    out_json = {
        "images": split_images,
        "annotations": split_anns,
        "categories": data["categories"]
    }

    out_json_path = os.path.join(split_dir, "_annotations.coco.json")
    with open(out_json_path, "w") as f:
        json.dump(out_json, f, indent=2)

print("\n✅ Done! COCO-style splits created in:")
for s in splits:
    print(f"  - {os.path.join(OUTPUT_ROOT, s)}")



Processing split: train (170 images)


100%|██████████| 170/170 [00:00<00:00, 564.32it/s]



Processing split: val (48 images)


100%|██████████| 48/48 [00:00<00:00, 533.29it/s]



Processing split: test (26 images)


100%|██████████| 26/26 [00:00<00:00, 530.60it/s]



✅ Done! COCO-style splits created in:
  - ../../../datasets/CRADataset\train
  - ../../../datasets/CRADataset\val
  - ../../../datasets/CRADataset\test
